In [9]:
import csv
import io
import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

CLASS_TO_IDX = {"fake": 0, "real": 1}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@dataclass
class Config:
    data_root: str
    train_csv: str
    valid_csv: str
    test_csv: str
    output_dir: str
    model_name: str
    image_size: int
    batch_size: int
    epochs: int
    learning_rate: float
    weight_decay: float
    num_workers: int
    patience: int
    label_smoothing: float
    dropout: float
    seed: int
    freeze_backbone_epochs: int

class RandomJPEGCompression:
    def __init__(self, quality_range=(45, 95), p=0.6):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, image: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return image
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

class DeepfakeDataset(Dataset):
    def __init__(self, csv_path: Path, data_root: Path, transform=None) -> None:
        self.data_root = data_root
        self.transform = transform
        self.samples = self._load_samples(csv_path)

    def _load_samples(self, csv_path: Path) -> List[Tuple[Path, int]]:
        samples = []
        with csv_path.open("r", newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                label_name = row["label_str"].strip().lower()
                relative_path = Path(*row["path"].split("/"))
                image_path = self.data_root / relative_path
                if not image_path.exists():
                    raise FileNotFoundError(f"Missing image referenced by {csv_path}: {image_path}")
                samples.append((image_path, CLASS_TO_IDX[label_name]))
        return samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        image_path, label = self.samples[index]
        with Image.open(image_path) as image:
            image = image.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

def build_transforms(image_size: int):
    train_transform = transforms.Compose([
        transforms.Resize((image_size + 24, image_size + 24)),
        transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_transform, eval_transform

def create_model(model_name: str, dropout: float) -> nn.Module:
    if model_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
    elif model_name == "resnet34":
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        in_features = model.fc.in_features
    elif model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = model.fc.in_features
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(p=dropout), nn.Linear(256, 64), nn.ReLU(), nn.Dropout(p=dropout), nn.Linear(64, 2))
    return model

def freeze_backbone(model: nn.Module, freeze: bool) -> None:
    for name, param in model.named_parameters():
        if not name.startswith("fc."):
            param.requires_grad = not freeze

def make_loader(dataset: Dataset, batch_size: int, num_workers: int, shuffle: bool) -> DataLoader:
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=num_workers > 0)

def accuracy_score_manual(labels: List[int], predictions: List[int]) -> float:
    correct = sum(int(label == pred) for label, pred in zip(labels, predictions))
    return correct / max(len(labels), 1)

def confusion_matrix_manual(labels: List[int], predictions: List[int]) -> List[List[int]]:
    tn = fp = fn = tp = 0
    for label, pred in zip(labels, predictions):
        if label == 0 and pred == 0:
            tn += 1
        elif label == 0 and pred == 1:
            fp += 1
        elif label == 1 and pred == 0:
            fn += 1
        else:
            tp += 1
    return [[tn, fp], [fn, tp]]

def binary_precision_recall_f1(labels: List[int], predictions: List[int], positive_label: int) -> Dict[str, float]:
    tp = fp = fn = 0
    support = 0
    for label, pred in zip(labels, predictions):
        if label == positive_label:
            support += 1
            if pred == positive_label:
                tp += 1
            else:
                fn += 1
        elif pred == positive_label:
            fp += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1-score": f1, "support": support}

def classification_report_manual(labels: List[int], predictions: List[int]) -> Dict[str, Dict[str, float]]:
    fake_metrics = binary_precision_recall_f1(labels, predictions, positive_label=0)
    real_metrics = binary_precision_recall_f1(labels, predictions, positive_label=1)
    accuracy = accuracy_score_manual(labels, predictions)
    macro_precision = (fake_metrics["precision"] + real_metrics["precision"]) / 2
    macro_recall = (fake_metrics["recall"] + real_metrics["recall"]) / 2
    macro_f1 = (fake_metrics["f1-score"] + real_metrics["f1-score"]) / 2
    total_support = fake_metrics["support"] + real_metrics["support"]
    weighted_precision = ((fake_metrics["precision"] * fake_metrics["support"]) + (real_metrics["precision"] * real_metrics["support"])) / max(total_support, 1)
    weighted_recall = ((fake_metrics["recall"] * fake_metrics["support"]) + (real_metrics["recall"] * real_metrics["support"])) / max(total_support, 1)
    weighted_f1 = ((fake_metrics["f1-score"] * fake_metrics["support"]) + (real_metrics["f1-score"] * real_metrics["support"])) / max(total_support, 1)
    return {
        "fake": fake_metrics,
        "real": real_metrics,
        "accuracy": accuracy,
        "macro avg": {"precision": macro_precision, "recall": macro_recall, "f1-score": macro_f1, "support": total_support},
        "weighted avg": {"precision": weighted_precision, "recall": weighted_recall, "f1-score": weighted_f1, "support": total_support},
    }

def roc_auc_score_manual(labels: List[int], scores: List[float]) -> float:
    positives = sum(labels)
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        return 0.0
    order = sorted(range(len(scores)), key=lambda idx: scores[idx])
    ranks = [0.0] * len(scores)
    current_rank = 1
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg_rank = (current_rank + current_rank + (j - i)) / 2
        for k in range(i, j + 1):
            ranks[order[k]] = avg_rank
        current_rank += j - i + 1
        i = j + 1
    positive_rank_sum = sum(ranks[idx] for idx, label in enumerate(labels) if label == 1)
    auc = (positive_rank_sum - (positives * (positives + 1) / 2)) / (positives * negatives)
    return float(auc)

def roc_curve_points_manual(labels: List[int], scores: List[float]):
    paired = sorted(zip(scores, labels), key=lambda item: item[0], reverse=True)
    positives = sum(labels)
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        return {"fpr": [0.0, 1.0], "tpr": [0.0, 1.0]}
    tp = 0
    fp = 0
    fpr = [0.0]
    tpr = [0.0]
    for score, label in paired:
        if label == 1:
            tp += 1
        else:
            fp += 1
        fpr.append(fp / negatives)
        tpr.append(tp / positives)
    return {"fpr": fpr, "tpr": tpr}

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device):
    model.eval()
    losses, all_labels, all_predictions, all_probs = [], [], [], []
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)
        losses.append(loss.item())
        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(preds.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())
    roc_curve = roc_curve_points_manual(all_labels, all_probs)
    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score_manual(all_labels, all_predictions),
        "f1": binary_precision_recall_f1(all_labels, all_predictions, positive_label=1)["f1-score"],
        "roc_auc": roc_auc_score_manual(all_labels, all_probs),
        "roc_curve": roc_curve,
        "confusion_matrix": confusion_matrix_manual(all_labels, all_predictions),
        "classification_report": classification_report_manual(all_labels, all_predictions),
    }

def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, scaler, device: torch.device):
    model.train()
    losses, all_labels, all_predictions = [], [], []
    autocast_enabled = device.type == "cuda"
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=autocast_enabled):
            logits = model(images)
            loss = criterion(logits, labels)
        if scaler is not None and autocast_enabled:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        preds = torch.argmax(logits, dim=1)
        losses.append(loss.item())
        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(preds.cpu().tolist())
    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score_manual(all_labels, all_predictions),
        "f1": binary_precision_recall_f1(all_labels, all_predictions, positive_label=1)["f1-score"],
    }

def save_checkpoint(checkpoint_path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, best_metric: float, config: Config) -> None:
    checkpoint = {"epoch": epoch, "best_metric": best_metric, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "config": asdict(config), "class_to_idx": CLASS_TO_IDX}
    torch.save(checkpoint, checkpoint_path)

def run_training(config: Config):
    set_seed(config.seed)
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    data_root = Path(config.data_root)
    train_transform, eval_transform = build_transforms(config.image_size)
    train_dataset = DeepfakeDataset(Path(config.train_csv), data_root, transform=train_transform)
    valid_dataset = DeepfakeDataset(Path(config.valid_csv), data_root, transform=eval_transform)
    test_dataset = DeepfakeDataset(Path(config.test_csv), data_root, transform=eval_transform)
    train_loader = make_loader(train_dataset, config.batch_size, config.num_workers, shuffle=True)
    valid_loader = make_loader(valid_dataset, config.batch_size, config.num_workers, shuffle=False)
    test_loader = make_loader(test_dataset, config.batch_size, config.num_workers, shuffle=False)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = create_model(config.model_name, config.dropout).to(device)
    freeze_backbone(model, freeze=True)
    criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    history = []
    best_f1 = -1.0
    patience_counter = 0
    checkpoint_path = output_dir / "best_model.pt"
    device_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
    print(f"Using device: {device} ({device_name})")
    print(f"Training samples: {len(train_dataset)} | Validation samples: {len(valid_dataset)} | Test samples: {len(test_dataset)}")
    for epoch in range(1, config.epochs + 1):
        # Backbone stays frozen for the full run; only the classifier head trains.
        print(f"\nEpoch {epoch}/{config.epochs}")
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
        valid_metrics = evaluate(model, valid_loader, criterion, device)
        scheduler.step(valid_metrics["f1"])
        history.append({"epoch": epoch, "train": train_metrics, "valid": {key: value for key, value in valid_metrics.items() if key not in {"classification_report", "confusion_matrix", "roc_curve"}}})
        print(f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} train_f1={train_metrics['f1']:.4f} val_loss={valid_metrics['loss']:.4f} val_acc={valid_metrics['accuracy']:.4f} val_f1={valid_metrics['f1']:.4f} val_auc={valid_metrics['roc_auc']:.4f}")
        if valid_metrics["f1"] > best_f1:
            best_f1 = valid_metrics["f1"]
            patience_counter = 0
            save_checkpoint(checkpoint_path, model, optimizer, epoch, best_f1, config)
            with (output_dir / "best_valid_metrics.json").open("w", encoding="utf-8") as handle:
                json.dump(valid_metrics, handle, indent=2)
            with (output_dir / "best_valid_roc_curve.json").open("w", encoding="utf-8") as handle:
                json.dump(valid_metrics["roc_curve"], handle, indent=2)
        else:
            patience_counter += 1
        with (output_dir / "history.json").open("w", encoding="utf-8") as handle:
            json.dump(history, handle, indent=2)
        if patience_counter >= config.patience:
            print(f"Early stopping triggered after {epoch} epochs.")
            break
    print("\nLoading best checkpoint for final test evaluation...")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    test_metrics = evaluate(model, test_loader, criterion, device)
    with (output_dir / "test_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(test_metrics, handle, indent=2)
    with (output_dir / "best_model_roc_curve.json").open("w", encoding="utf-8") as handle:
        json.dump(test_metrics["roc_curve"], handle, indent=2)
    with (output_dir / "config.json").open("w", encoding="utf-8") as handle:
        json.dump(asdict(config), handle, indent=2)
    print("\nFinal test metrics")
    print(json.dumps({k: v for k, v in test_metrics.items() if k not in {"classification_report", "confusion_matrix", "roc_curve"}}, indent=2))
    print(f"Best model saved to: {checkpoint_path}")


In [10]:
project_root = Path.cwd()
data_root = project_root / "real_vs_fake" / "real-vs-fake"

print(f"Project root: {project_root}")
print(f"Dataset exists: {data_root.exists()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: c:\Vivek\code\deepfakedetection
Dataset exists: True
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [11]:
config = Config(
    data_root="real_vs_fake/real-vs-fake",
    train_csv="train.csv",
    valid_csv="valid.csv",
    test_csv="test.csv",
    output_dir="artifacts",
    model_name="resnet18",   # choices: resnet18, resnet34, resnet50
    image_size=224,
    batch_size=32,
    epochs=10,
    learning_rate=3e-4,
    weight_decay=1e-4,
    num_workers=0,
    patience=3,
    label_smoothing=0.05,
    dropout=0.3,
    seed=42,
    freeze_backbone_epochs=10,
)

config

Config(data_root='real_vs_fake/real-vs-fake', train_csv='train.csv', valid_csv='valid.csv', test_csv='test.csv', output_dir='artifacts', model_name='resnet18', image_size=224, batch_size=32, epochs=10, learning_rate=0.0003, weight_decay=0.0001, num_workers=0, patience=3, label_smoothing=0.05, dropout=0.3, seed=42, freeze_backbone_epochs=10)

In [12]:
run_training(config)

Using device: cuda (NVIDIA GeForce RTX 4050 Laptop GPU)
Training samples: 100000 | Validation samples: 20000 | Test samples: 20000

Epoch 1/10


train_loss=0.5607 train_acc=0.7274 train_f1=0.7310 val_loss=0.4915 val_acc=0.7810 val_f1=0.7701 val_auc=0.8687

Epoch 2/10


train_loss=0.5285 train_acc=0.7535 train_f1=0.7546 val_loss=0.4811 val_acc=0.7889 val_f1=0.7675 val_auc=0.8847

Epoch 3/10


train_loss=0.5197 train_acc=0.7617 train_f1=0.7619 val_loss=0.4591 val_acc=0.8117 val_f1=0.8010 val_auc=0.8973

Epoch 4/10


train_loss=0.5112 train_acc=0.7686 train_f1=0.7688 val_loss=0.4479 val_acc=0.8153 val_f1=0.8165 val_auc=0.8982

Epoch 5/10


...

train_loss=0.4927 train_acc=0.7828 train_f1=0.7831 val_loss=0.4287 val_acc=0.8283 val_f1=0.8228 val_auc=0.9107

Epoch 8/10


train_loss=0.4889 train_acc=0.7847 train_f1=0.7839 val_loss=0.4169 val_acc=0.8368 val_f1=0.8368 val_auc=0.9170

Epoch 9/10


train_loss=0.4870 train_acc=0.7873 train_f1=0.7866 val_loss=0.4197 val_acc=0.8364 val_f1=0.8364 val_auc=0.9172

Epoch 10/10


train_loss=0.4869 train_acc=0.7863 train_f1=0.7865 val_loss=0.4243 val_acc=0.8344 val_f1=0.8250 val_auc=0.9193

Loading best checkpoint for final test evaluation...



Final test metrics
{
  "loss": 0.4178325711250305,
  "accuracy": 0.83755,
  "f1": 0.8380116667497632,
  "roc_auc": 0.916002695
}
Best model saved to: artifacts\best_model.pt
